# Task 2 — Data Discovery, Profiling and Cleaning
**Project:** SkyPrint — Flight Tracking & Climate Impact Analysis

Inputs (from Task 1, `data/raw/`):
- `opensky.json` — live aircraft state vectors from the OpenSky Network `/states/all` endpoint
- `openmeteo.json` — hourly weather data from the Open-Meteo API
- `aircraftDatabase.csv` — aircraft type lookup (plane_id → manufacturer/model)

this notebook profiles and cleans each separately and writes three interim files:
- `data/interim/cleaned_opensky.csv`
- `data/interim/cleaned_weather.csv`
- `data/interim/cleaned_aircraft.csv`

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

RAW_DIR = Path('../data/raw')
INTERIM_DIR = Path('../data/interim')
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)

## 1. OpenSky — Load and Flatten

`/states/all` returns a JSON object with a `time` field and a `states` field, which is a **list of lists** with no column names attached — the schema is fixed and documented by OpenSky. We define it explicitly.

In [2]:
# Task 1 now saves dated raw files (data/raw/opensky_<date>.json); pick the most recent one
opensky_files = sorted(RAW_DIR.glob('opensky_*.json'))
opensky_path = opensky_files[-1] if opensky_files else RAW_DIR / 'opensky.json'

with open(opensky_path) as f:
    opensky_raw = json.load(f)

print('loaded file:', opensky_path.name)
print('snapshot unix time:', opensky_raw['time'])
print('number of state vectors:', len(opensky_raw['states']))
opensky_raw['states'][0]

loaded file: opensky_2026-09-12.json
snapshot unix time: 1789212388
number of state vectors: 8680


['39de4f',
 'TVF95DU ',
 'France',
 1789212387,
 1789212387,
 -3.2193,
 40.9777,
 5867.4,
 False,
 181.5,
 9.46,
 11.38,
 None,
 6301.74,
 '1000',
 False,
 0,
 0]

In [3]:
# Official OpenSky /states/all column order (18 fields — 'category' requires the extended=1
# query parameter on the Task 1 request, which is now in place)
OPENSKY_COLUMNS = [
    'plane_id', 'flight_id', 'origin_country', 'time_position', 'last_contact',
    'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity',
    'true_track', 'vertical_rate', 'sensors', 'geo_altitude', 'squawk',
    'spi', 'source_type', 'category'
]

df_sky = pd.DataFrame(opensky_raw['states'], columns=OPENSKY_COLUMNS)
df_sky.shape

(8680, 18)

In [4]:
df_sky.head()

,plane_id,flight_id,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,source_type,category
0,39de4f,TVF95DU,France,1.789212e+09,1789212387,-3.2193,40.9777,5867.40,False,181.50,9.46,11.38,None,6301.74,1000,False,0,0
1,39de4e,TVF10HW,France,1.789212e+09,1789212387,25.0895,36.5625,11887.20,False,256.70,111.27,0.00,None,12565.38,0664,False,0,0
2,80162c,AXB864,India,1.789212e+09,1789212387,54.0766,25.8006,11887.20,False,243.92,107.17,0.00,None,12679.68,3103,False,0,0
3,e8027e,LPE2423,Chile,1.789212e+09,1789212387,-54.5804,-26.7547,10972.80,False,175.05,310.95,0.00,None,11452.86,NaN,False,0,0
4,e8027d,LPE2352,Chile,1.789212e+09,1789212387,-76.8829,-11.9881,3840.48,False,181.15,331.69,10.73,None,4084.32,NaN,False,0,0


## 2. OpenSky — Profiling

In [5]:
def profile(df):
    rows = []
    for c in df.columns:
        s = df[c]
        sample = s.dropna().iloc[0] if s.notna().any() else None
        rows.append({
            'column': c,
            'dtype': str(s.dtype),
            'null_count': int(s.isna().sum()),
            'percent_null': round(100 * s.isna().sum() / len(s), 2),
            'unique_count': int(s.nunique(dropna=True)),
            'sample_value': sample
        })
    return pd.DataFrame(rows)

profile_sky = profile(df_sky)
profile_sky

,column,dtype,null_count,percent_null,unique_count,sample_value
0,plane_id,str,0,0.00,8680,39de4f
1,flight_id,str,0,0.00,8490,TVF95DU
2,origin_country,str,0,0.00,109,France
3,time_position,float64,113,1.30,377,1789212387.0
4,last_contact,int64,0,0.00,298,1789212387
5,longitude,float64,113,1.30,8511,-3.2193
6,latitude,float64,113,1.30,8417,40.9777
7,baro_altitude,float64,909,10.47,1387,5867.4
8,on_ground,bool,0,0.00,2,False
9,velocity,float64,3,0.03,5982,181.5


In [6]:
print('rows:', len(df_sky), ' columns:', df_sky.shape[1])
print('exact duplicate rows:', df_sky.duplicated().sum())
print('duplicate (plane_id, time_position) pairs:', df_sky.duplicated(subset=['plane_id','time_position']).sum())
print('empty-string flight_ids:', (df_sky['flight_id'].str.strip() == '').sum())
print('category value counts:')
print(df_sky['category'].value_counts(dropna=False).sort_index())

rows: 8680  columns: 18
exact duplicate rows: 0
duplicate (plane_id, time_position) pairs: 0
empty-string flight_ids: 187
category value counts:
category
0     8256
1      202
2       10
3        7
4      115
5        3
6       78
8        1
12       1
13       7
Name: count, dtype: int64


### Issues found — OpenSky

| # | Issue | Decision |
|---|---|---|
| 1 | `sensors` is null for 100% of rows (it only applies to ADS-B ground-sensor filtered queries, unused here) | Drop the column |
| 2 | `flight_id` has trailing whitespace padding (fixed-width field from the API, e.g. `'TVF13SR '`) and some are empty strings for aircraft without an assigned flight_id | Strip whitespace; convert empty strings to `NaN` |
| 3 | `time_position` and `longitude`/`latitude` are null together for the same ~139 rows (no position report received in this update cycle) | Keep as `NaN`, do not impute — this is a genuine missing-position case, not a data-entry problem |
| 4 | `baro_altitude`, `geo_altitude`, `vertical_rate` are null mainly for aircraft on the ground (`on_ground = True`) or with incomplete state reports | Keep as `NaN`; document that null altitude is expected for grounded aircraft |
| 5 | `squawk` is not needed for this project's analysis (transponder code, unrelated to flight-tracking or fuel/climate questions) | Drop the column, per team decision |
| 6 | `spi` is not needed for this project's analysis (special-position-indicator flag, no data-quality problem, just out of scope) | Drop the column, per team decision |
| 7 | `time_position` / `last_contact` are Unix epoch integers, not human-readable | Convert to UTC `datetime64` |
| 8 | No duplicate `(plane_id, time_position)` pairs found in this snapshot | No action needed, but the dedup step is kept in the pipeline for future runs |
| 9 | Original field name `position_source` was ambiguous on review | Renamed to `source_type` for clarity |
| 10 | `category` requires `extended=1` on the Task 1 API request — the raw file now includes it as an 18th field, all values fall inside the documented 0–13 range with no nulls in this snapshot | Keep as-is (`int`); no cleaning needed |


## 3. OpenSky — Cleaning

In [7]:
df_sky_clean = df_sky.copy()

# 1) drop fully-null / out-of-scope columns
df_sky_clean = df_sky_clean.drop(columns=['sensors', 'squawk', 'spi'])

# 2) standardise flight_id text
df_sky_clean['flight_id'] = df_sky_clean['flight_id'].str.strip()
df_sky_clean.loc[df_sky_clean['flight_id'] == '', 'flight_id'] = np.nan

# 3) fix data types — epoch seconds -> UTC datetime
for col in ['time_position', 'last_contact']:
    df_sky_clean[col] = pd.to_datetime(df_sky_clean[col], unit='s', utc=True)

# 4) make sure booleans are true bool dtype
df_sky_clean['on_ground'] = df_sky_clean['on_ground'].astype('boolean')

# 4b) category is a categorical code (0-13 documented range) — nullable integer, not float
df_sky_clean['category'] = df_sky_clean['category'].astype('Int64')

# 5) drop duplicates on the natural key
before = len(df_sky_clean)
df_sky_clean = df_sky_clean.drop_duplicates(subset=['plane_id', 'time_position'])
print(f'dropped {before - len(df_sky_clean)} duplicate rows')

# 6) columns are already snake_case
df_sky_clean.columns = [c.lower() for c in df_sky_clean.columns]

df_sky_clean.dtypes

dropped 0 duplicate rows


plane_id                         str
flight_id                        str
origin_country                   str
time_position     datetime64[s, UTC]
last_contact      datetime64[s, UTC]
longitude                    float64
latitude                     float64
baro_altitude                float64
on_ground                    boolean
velocity                     float64
true_track                   float64
vertical_rate                float64
geo_altitude                 float64
source_type                    int64
category                       Int64
dtype: object

In [8]:
df_sky_clean.head()

,plane_id,flight_id,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,source_type,category
0,39de4f,TVF95DU,France,2026-09-12 11:26:27+00:00,2026-09-12 11:26:27+00:00,-3.2193,40.9777,5867.40,False,181.50,9.46,11.38,6301.74,0,0
1,39de4e,TVF10HW,France,2026-09-12 11:26:27+00:00,2026-09-12 11:26:27+00:00,25.0895,36.5625,11887.20,False,256.70,111.27,0.00,12565.38,0,0
2,80162c,AXB864,India,2026-09-12 11:26:27+00:00,2026-09-12 11:26:27+00:00,54.0766,25.8006,11887.20,False,243.92,107.17,0.00,12679.68,0,0
3,e8027e,LPE2423,Chile,2026-09-12 11:26:27+00:00,2026-09-12 11:26:27+00:00,-54.5804,-26.7547,10972.80,False,175.05,310.95,0.00,11452.86,0,0
4,e8027d,LPE2352,Chile,2026-09-12 11:26:27+00:00,2026-09-12 11:26:27+00:00,-76.8829,-11.9881,3840.48,False,181.15,331.69,10.73,4084.32,0,0


In [9]:
df_sky_clean.to_csv(INTERIM_DIR / 'cleaned_opensky.csv', index=False)
print('saved:', (INTERIM_DIR / 'cleaned_opensky.csv').resolve())

saved: /home/claude/DE-SkyPrint-v3/data/interim/cleaned_opensky.csv


## 4. Open-Meteo — Load and Flatten

The hourly block is a set of **parallel arrays** (`time`, `temperature_2m`, ... all the same length) rather than a list of records, so we build the DataFrame directly from `hourly`, then attach the request-level metadata (`latitude`, `longitude`, `elevation`, `timezone`) as constant columns.

In [10]:
# same dated-file situation as OpenSky — pick the most recent openmeteo_<date>.json
meteo_files = sorted(RAW_DIR.glob('openmeteo_*.json'))
meteo_path = meteo_files[-1] if meteo_files else RAW_DIR / 'openmeteo.json'

with open(meteo_path) as f:
    meteo_raw = json.load(f)

print('loaded file:', meteo_path.name)
print(meteo_raw['hourly_units'])
df_wx = pd.DataFrame(meteo_raw['hourly'])
df_wx['latitude'] = meteo_raw['latitude']
df_wx['longitude'] = meteo_raw['longitude']
df_wx['elevation_m'] = meteo_raw['elevation']
df_wx['timezone'] = meteo_raw['timezone']
df_wx.shape

loaded file: openmeteo_2026-09-12.json
{'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h', 'wind_direction_10m': '°', 'surface_pressure': 'hPa'}


(168, 10)

In [11]:
df_wx.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,latitude,longitude,elevation_m,timezone
0,2026-09-12T00:00,34.0,20,5.4,340,939.4,24.710016,46.688103,634.0,GMT
1,2026-09-12T01:00,33.0,21,6.1,332,939.4,24.710016,46.688103,634.0,GMT
2,2026-09-12T02:00,32.0,22,6.8,348,939.6,24.710016,46.688103,634.0,GMT
3,2026-09-12T03:00,31.5,23,6.6,17,939.9,24.710016,46.688103,634.0,GMT
4,2026-09-12T04:00,32.9,21,4.6,36,940.8,24.710016,46.688103,634.0,GMT


## 5. Open-Meteo — Profiling

In [12]:
profile_wx = profile(df_wx)
profile_wx

,column,dtype,null_count,percent_null,unique_count,sample_value
0,time,str,0,0.0,168,2026-09-12T00:00
1,temperature_2m,float64,0,0.0,100,34.0
2,relative_humidity_2m,int64,0,0.0,17,20
3,wind_speed_10m,float64,0,0.0,84,5.4
4,wind_direction_10m,int64,0,0.0,124,340
5,surface_pressure,float64,0,0.0,43,939.4
6,latitude,float64,0,0.0,1,24.710016
7,longitude,float64,0,0.0,1,46.688103
8,elevation_m,float64,0,0.0,1,634.0
9,timezone,str,0,0.0,1,GMT


In [13]:
print('rows:', len(df_wx))
print('exact duplicate rows:', df_wx.duplicated().sum())
print('duplicate timestamps:', df_wx.duplicated(subset=['time']).sum())

rows: 168
exact duplicate rows: 0
duplicate timestamps: 0


### Issues found — Open-Meteo

| # | Issue | Decision |
|---|---|---|
| 1 | `time` is an ISO-8601 string, not a real datetime | Convert to `datetime64` |
| 2 | `latitude`/`longitude`/`elevation`/`timezone` are single scalars in the raw response, not arrays — flattening them naively with `json_normalize` would only keep one row | Broadcast them as constant columns across every hourly row instead |
| 3 | No nulls or duplicate timestamps found in this pull | No cleaning action needed, checks kept in the pipeline for future runs |
| 4 | Column names are already snake_case (`temperature_2m`, `wind_speed_10m`, ...) | No renaming needed |


## 6. Open-Meteo — Cleaning

In [14]:
df_wx_clean = df_wx.copy()
df_wx_clean['time'] = pd.to_datetime(df_wx_clean['time'])
df_wx_clean = df_wx_clean.drop_duplicates(subset=['time'])
df_wx_clean = df_wx_clean.rename(columns={'time': 'observation_time'})
df_wx_clean.dtypes

observation_time        datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
wind_speed_10m                 float64
wind_direction_10m               int64
surface_pressure               float64
latitude                       float64
longitude                      float64
elevation_m                    float64
timezone                           str
dtype: object

In [15]:
df_wx_clean.to_csv(INTERIM_DIR / 'cleaned_weather.csv', index=False)
print('saved:', (INTERIM_DIR / 'cleaned_weather.csv').resolve())

saved: /home/claude/DE-SkyPrint-v3/data/interim/cleaned_weather.csv


## 7. Aircraft Type Lookup — Load and Profile

`aircraftDatabase.csv` maps `plane_id` (the same 24-bit aircraft address used in the OpenSky feed) to manufacturer, model, and aircraft type. It's not joined here — that join happens in Task 4 — but it goes through the same profiling and cleaning pass as the other two sources so it's ready to use.

In [16]:
df_ac = pd.read_csv(RAW_DIR / 'aircraftDatabase.csv', dtype=str)
# the source file's header is 'icao24' — rename immediately so it matches the OpenSky side
df_ac = df_ac.rename(columns={'icao24': 'plane_id'})
df_ac.shape

(520000, 27)

In [17]:
df_ac.head(3)

,plane_id,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,operatorcallsign,operatoricao,operatoriata,owner,testreg,registered,reguntil,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,false,false,false,NaN,NaN
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,NaN,L1P,NaN,NaN,NaN,NaN,Vintage Aircraft Llc,NaN,NaN,2027-01-31,NaN,NaN,NaN,NaN,NaN,false,false,false,NaN,NaN
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,NaN,L2P,NaN,NaN,NaN,NaN,Tvpx Aircraft Solutions Inc Trustee,NaN,NaN,2029-08-31,NaN,1977-01-01,NaN,NaN,LYCOMING TI0-540 SER,false,false,false,NaN,NaN


In [18]:
profile_ac = profile(df_ac)
profile_ac

,column,dtype,null_count,percent_null,unique_count,sample_value
0,plane_id,str,1,0.00,519997,aa3487
1,registration,str,3474,0.67,514254,N757F
2,manufacturericao,str,91199,17.54,739,RAYTHEON
3,manufacturername,str,81434,15.66,41540,Raytheon Aircraft Company
4,model,str,79574,15.30,34482,A36
5,typecode,str,40053,7.70,1933,BE36
6,serialnumber,str,82882,15.94,295525,E-3121
7,linenumber,str,519029,99.81,693,AF1085
8,icaoaircrafttype,str,91216,17.54,51,L1P
9,operator,str,496302,95.44,4001,Indian Air Force


In [19]:
null_pct = (df_ac.isna().sum() / len(df_ac) * 100).round(2).sort_values(ascending=False)
print('rows with plane_id null:', df_ac['plane_id'].isna().sum())
print('exact duplicate rows:', df_ac.duplicated().sum())
print('duplicate plane_id (non-null):', df_ac[df_ac['plane_id'].notna()].duplicated(subset=['plane_id']).sum())
print('modes unique values:', df_ac['modes'].unique())
print('adsb unique values:', df_ac['adsb'].unique())
print('acars unique values:', df_ac['acars'].unique())
print()
print('columns with >= 90% nulls (for reference — the columns dropped below are the fixed list from this check):')
print(null_pct[null_pct >= 90])

rows with plane_id null: 1


exact duplicate rows: 2


duplicate plane_id (non-null): 2
modes unique values: <StringArray>
['false']
Length: 1, dtype: str
adsb unique values: <StringArray>
['false']
Length: 1, dtype: str
acars unique values: <StringArray>
['false']
Length: 1, dtype: str

columns with >= 90% nulls (for reference — the columns dropped below are the fixed list from this check):
status                 100.00
seatconfiguration      100.00
firstflightdate         99.94
testreg                 99.94
notes                   99.93
linenumber              99.81
categoryDescription     98.61
operatoriata            98.45
operator                95.44
operatorcallsign        92.29
operatoricao            92.04
dtype: float64


### Issues found — Aircraft Database

| # | Issue | Decision |
|---|---|---|
| 1 | 1 row has `plane_id` null — it carries no other identifying info either, it's a junk row | Drop the row |
| 2 | 2 `plane_id` values appear as exact duplicate rows (e.g. `ae690b`, `ae6963`) | Drop duplicates, keep first occurrence |
| 3 | `modes`, `adsb`, `acars` are `'false'` for every single row — zero variance, no information content | Drop all three columns |
| 4 | 11 columns are ≥90% null: `status` (100%), `seatconfiguration` (100%), `firstflightdate` (99.94%), `testreg` (99.94%), `notes` (99.93%), `linenumber` (99.81%), `categoryDescription` (98.61%), `operatoriata` (98.45%), `operator` (95.44%), `operatorcallsign` (92.29%), `operatoricao` (92.04%) | Drop these 11 named columns, per team decision |
| 5 | `built`, `registered`, `reguntil` are date strings stored as text | Convert to `datetime64` |


## 8. Aircraft Type Lookup — Cleaning

In [20]:
df_ac_clean = df_ac.copy()

# 1) drop the row with no plane_id at all
df_ac_clean = df_ac_clean.dropna(subset=['plane_id'])

# 2) drop exact duplicate rows
before = len(df_ac_clean)
df_ac_clean = df_ac_clean.drop_duplicates()
print(f'dropped {before - len(df_ac_clean)} duplicate rows')

# 3) drop zero-variance flag columns + the 11 columns identified as >= 90% null (fixed list, not recomputed)
cols_to_drop = [
    'modes', 'adsb', 'acars',
    'status', 'seatconfiguration', 'firstflightdate', 'testreg', 'notes',
    'linenumber', 'categoryDescription', 'operatoriata', 'operator',
    'operatorcallsign', 'operatoricao'
]
df_ac_clean = df_ac_clean.drop(columns=cols_to_drop)
print('dropped columns:', cols_to_drop)

# 4) standardise plane_id casing (hex codes should be consistent lowercase, matches OpenSky's format)
df_ac_clean['plane_id'] = df_ac_clean['plane_id'].str.strip().str.lower()

# 5) parse date-like text columns that survived the drop
for col in ['built', 'registered', 'reguntil']:
    df_ac_clean[col] = pd.to_datetime(df_ac_clean[col], errors='coerce')

df_ac_clean.dtypes

dropped 2 duplicate rows
dropped columns: ['modes', 'adsb', 'acars', 'status', 'seatconfiguration', 'firstflightdate', 'testreg', 'notes', 'linenumber', 'categoryDescription', 'operatoriata', 'operator', 'operatorcallsign', 'operatoricao']


plane_id                       str
registration                   str
manufacturericao               str
manufacturername               str
model                          str
typecode                       str
serialnumber                   str
icaoaircrafttype               str
owner                          str
registered          datetime64[us]
reguntil            datetime64[us]
built               datetime64[us]
engines                        str
dtype: object

In [21]:
df_ac_clean.head()

,plane_id,registration,manufacturericao,manufacturername,model,typecode,serialnumber,icaoaircrafttype,owner,registered,reguntil,built,engines
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,L1P,Vintage Aircraft Llc,NaT,2027-01-31,NaT,NaN
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,L2P,Tvpx Aircraft Solutions Inc Trustee,NaT,2029-08-31,1977-01-01,LYCOMING TI0-540 SER
3,a7a809,N5926K,ROCKWELL,NaN,NaN,AC90,NaN,L2T,NaN,NaT,NaT,NaT,NaN
4,391927,F-GGJH,ROBIN,Robin,DR.400 160 Chevalier,DR40,1795,L1P,Private,NaT,NaT,NaT,NaN
5,503c21,LY-KNA,NaN,Impulse Aircraft,Impulse 100,ZZZZ,NaN,NaN,Private,NaT,NaT,NaT,NaN


In [22]:
df_ac_clean.to_csv(INTERIM_DIR / 'cleaned_aircraft.csv', index=False)
print('saved:', (INTERIM_DIR / 'cleaned_aircraft.csv').resolve())

saved: /home/claude/DE-SkyPrint-v3/data/interim/cleaned_aircraft.csv


## 9. Summary

In [23]:
summary = pd.DataFrame([
    {'dataset': 'OpenSky states', 'raw_rows': len(df_sky), 'cleaned_rows': len(df_sky_clean),
     'columns_dropped': 'sensors, squawk, spi', 'output': 'data/interim/cleaned_opensky.csv'},
    {'dataset': 'Open-Meteo hourly', 'raw_rows': len(df_wx), 'cleaned_rows': len(df_wx_clean),
     'columns_dropped': '-', 'output': 'data/interim/cleaned_weather.csv'},
    {'dataset': 'Aircraft type lookup', 'raw_rows': len(df_ac), 'cleaned_rows': len(df_ac_clean),
     'columns_dropped': f'{len(cols_to_drop)} columns (modes, adsb, acars + 11 columns >= 90% null)',
     'output': 'data/interim/cleaned_aircraft.csv'},
])
summary

,dataset,raw_rows,cleaned_rows,columns_dropped,output
0,OpenSky states,8680,8680,"sensors, squawk, spi",data/interim/cleaned_opensky.csv
1,Open-Meteo hourly,168,168,-,data/interim/cleaned_weather.csv
2,Aircraft type lookup,520000,519997,"14 columns (modes, adsb, acars + 11 columns >=...",data/interim/cleaned_aircraft.csv
